# 03 — Activations & Loss Functions from Scratch

**Goal:** Implement every major activation and loss function (forward + backward), visualize them, and build intuition for interview questions.

---

## Why Activations Matter

Without non-linear activations, a stack of linear layers collapses to a single linear transformation: $W_2(W_1 x + b_1) + b_2 = W' x + b'$. Activations introduce non-linearity that enables learning complex functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
%matplotlib inline

## 1. Activation Functions — Implementations

Each activation is a class with `forward(x)` and `backward(dout)` methods.

In [ ]:
class ReLU:
    """f(x) = max(0, x).  Gradient: 1 if x > 0, else 0."""
    def forward(self, x):
        self.mask = (x > 0).astype(float)
        return x * self.mask
    
    def backward(self, dout):
        return dout * self.mask


class LeakyReLU:
    """f(x) = x if x > 0 else alpha*x.  Fixes dead ReLU."""
    def __init__(self, alpha=0.01):
        self.alpha = alpha
    
    def forward(self, x):
        self.x = x
        return np.where(x > 0, x, self.alpha * x)
    
    def backward(self, dout):
        return dout * np.where(self.x > 0, 1.0, self.alpha)


class ELU:
    """f(x) = x if x > 0 else alpha*(exp(x)-1).  Smooth for x < 0."""
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    
    def forward(self, x):
        self.x = x
        self.out = np.where(x > 0, x, self.alpha * (np.exp(x) - 1))
        return self.out
    
    def backward(self, dout):
        return dout * np.where(self.x > 0, 1.0, self.out + self.alpha)


class GELU:
    """Gaussian Error Linear Unit: f(x) = x * Phi(x) where Phi is the standard normal CDF.
    Approximation: 0.5*x*(1 + tanh(sqrt(2/pi)*(x + 0.044715*x^3)))"""
    def forward(self, x):
        self.x = x
        self.inner = np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)
        self.tanh_inner = np.tanh(self.inner)
        return 0.5 * x * (1.0 + self.tanh_inner)
    
    def backward(self, dout):
        x = self.x
        sech2 = 1.0 - self.tanh_inner ** 2
        inner_grad = np.sqrt(2.0 / np.pi) * (1.0 + 3 * 0.044715 * x**2)
        return dout * (0.5 * (1.0 + self.tanh_inner) + 0.5 * x * sech2 * inner_grad)


class Swish:
    """SiLU/Swish: f(x) = x * sigmoid(x)"""
    def forward(self, x):
        self.x = x
        self.sig = 1.0 / (1.0 + np.exp(-x))
        return x * self.sig
    
    def backward(self, dout):
        return dout * (self.sig + self.x * self.sig * (1.0 - self.sig))


class Sigmoid:
    """f(x) = 1/(1+exp(-x)).  Gradient: f(x)*(1-f(x))."""
    def forward(self, x):
        self.out = 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
        return self.out
    
    def backward(self, dout):
        return dout * self.out * (1.0 - self.out)


class TanhActivation:
    """f(x) = tanh(x).  Gradient: 1 - tanh(x)^2."""
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out
    
    def backward(self, dout):
        return dout * (1.0 - self.out ** 2)


class Softmax:
    """Softmax: converts logits to probabilities. Usually fused with cross-entropy."""
    def forward(self, x):
        shifted = x - np.max(x, axis=-1, keepdims=True)
        exp_x = np.exp(shifted)
        self.out = exp_x / np.sum(exp_x, axis=-1, keepdims=True)
        return self.out
    
    def backward(self, dout):
        # Jacobian: diag(s) - s*s^T for each sample
        # For each sample i: dx_i = s_i * (dout_i - sum(dout_i * s_i))
        s = self.out
        return s * (dout - np.sum(dout * s, axis=-1, keepdims=True))


print("All activation functions defined.")

## 2. Visualize All Activations and Their Gradients

In [ ]:
x = np.linspace(-4, 4, 500)

activations = {
    'ReLU': ReLU(),
    'LeakyReLU (a=0.1)': LeakyReLU(0.1),
    'ELU': ELU(),
    'GELU': GELU(),
    'Swish/SiLU': Swish(),
    'Sigmoid': Sigmoid(),
    'Tanh': TanhActivation(),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for idx, (name, act) in enumerate(activations.items()):
    row, col = idx // 4, idx % 4
    ax = axes[row, col]
    
    y = act.forward(x.copy())
    grad = act.backward(np.ones_like(x))
    
    ax.plot(x, y, 'b-', linewidth=2, label='f(x)')
    ax.plot(x, grad, 'r--', linewidth=1.5, label="f'(x)")
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(-2, 4)
    ax.grid(True, alpha=0.3)

# Remove empty subplot
axes[1, 3].axis('off')

plt.suptitle('Activation Functions and Their Gradients', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. The Dead ReLU Problem

If a ReLU neuron's pre-activation is always negative (e.g., after a bad weight update), its gradient is permanently zero. The neuron "dies" and never learns again.

In [ ]:
# Simulate dead ReLU: track fraction of dead neurons during training
np.random.seed(42)
n_neurons = 256
n_samples = 100

# Random input
x = np.random.randn(n_samples, 128)

# Track dead neurons with different biases
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, (bias_init, title) in enumerate([
    (-2.0, 'Large negative bias (-2.0)'),
    (0.0, 'Zero bias (standard)'),
    (0.01, 'Small positive bias (0.01)')
]):
    dead_fracs = []
    # Simulate a 10-layer ReLU network
    h = x.copy()
    for layer_idx in range(10):
        W = np.random.randn(h.shape[1], n_neurons) * np.sqrt(2.0 / h.shape[1])
        b = np.full((1, n_neurons), bias_init)
        z = h @ W + b
        h = np.maximum(0, z)  # ReLU
        dead_frac = np.mean(np.all(h == 0, axis=0))  # neurons dead for ALL samples
        dead_fracs.append(dead_frac)
    
    ax = axes[idx]
    ax.bar(range(10), dead_fracs, color='coral', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Fraction of dead neurons')
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.suptitle('Dead ReLU Problem: Fraction of Permanently Dead Neurons', y=1.02)
plt.tight_layout()
plt.show()

print("Fix: use LeakyReLU/ELU/GELU, careful initialization, avoid large learning rates.")

## 4. Loss Functions — Implementations

### Key Formulas

| Loss | Formula | Gradient w.r.t. prediction |
|------|---------|---------------------------|
| MSE | $\frac{1}{N}\sum(y - \hat{y})^2$ | $\frac{2}{N}(\hat{y} - y)$ |
| Cross-Entropy | $-\frac{1}{N}\sum y_c \log \hat{y}_c$ | $\frac{1}{N}(\hat{y} - y)$ (with softmax) |
| BCE | $-\frac{1}{N}\sum [y\log\hat{y} + (1-y)\log(1-\hat{y})]$ | $\frac{1}{N}(\frac{-y}{\hat{y}} + \frac{1-y}{1-\hat{y}})$ |
| Focal | $-\alpha(1-\hat{y})^\gamma \log \hat{y}$ | focuses on hard examples |
| Huber | MSE if $|e|<\delta$, else MAE-like | smooth transition |

In [ ]:
class MSELoss:
    """Mean Squared Error: L = (1/N) * sum((y_pred - y_true)^2)"""
    def forward(self, pred, target):
        self.pred = pred
        self.target = target
        self.N = pred.shape[0]
        self.diff = pred - target
        return np.mean(self.diff ** 2)
    
    def backward(self):
        return 2.0 * self.diff / self.N


class CrossEntropyLoss:
    """Softmax + Negative Log-Likelihood (numerically stable, fused)."""
    def forward(self, logits, targets):
        """logits: (N, C), targets: (N,) integer labels."""
        self.N = logits.shape[0]
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        self.targets = targets
        log_probs = -np.log(self.probs[np.arange(self.N), targets] + 1e-12)
        return np.mean(log_probs)
    
    def backward(self):
        dlogits = self.probs.copy()
        dlogits[np.arange(self.N), self.targets] -= 1.0
        return dlogits / self.N


class BCELoss:
    """Binary Cross-Entropy (expects probabilities from sigmoid)."""
    def forward(self, pred, target):
        self.pred = np.clip(pred, 1e-7, 1 - 1e-7)
        self.target = target
        self.N = pred.shape[0]
        return -np.mean(target * np.log(self.pred) + (1 - target) * np.log(1 - self.pred))
    
    def backward(self):
        return (-(self.target / self.pred) + (1 - self.target) / (1 - self.pred)) / self.N


class FocalLoss:
    """Focal Loss: down-weights easy examples, focuses on hard ones.
    FL = -alpha * (1-p)^gamma * log(p) for correct class."""
    def __init__(self, gamma=2.0, alpha=1.0):
        self.gamma = gamma
        self.alpha = alpha
    
    def forward(self, logits, targets):
        self.N = logits.shape[0]
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        self.targets = targets
        
        p_t = self.probs[np.arange(self.N), targets]
        self.p_t = np.clip(p_t, 1e-7, 1.0)
        focal_weight = self.alpha * (1 - self.p_t) ** self.gamma
        loss = -focal_weight * np.log(self.p_t)
        return np.mean(loss)
    
    def backward(self):
        # Simplified gradient for softmax + focal loss
        p_t = self.p_t
        gamma = self.gamma
        alpha = self.alpha
        
        # dL/dp_t = alpha * [(1-p_t)^gamma / p_t - gamma * (1-p_t)^(gamma-1) * log(p_t)]
        # Then chain through softmax
        dprobs = self.probs.copy()
        focal_w = alpha * (1 - p_t) ** gamma
        focal_grad_factor = focal_w / p_t + alpha * gamma * (1 - p_t) ** (gamma - 1) * np.log(p_t)
        
        # Standard softmax gradient structure, weighted by focal term
        dlogits = self.probs.copy()
        dlogits[np.arange(self.N), self.targets] -= 1.0
        # Scale by focal weight (approximate — exact version is more complex)
        scale = focal_w.reshape(-1, 1)
        return dlogits * scale / self.N


class HuberLoss:
    """Huber Loss: MSE for small errors, MAE for large errors.
    Smooth transition at |error| = delta."""
    def __init__(self, delta=1.0):
        self.delta = delta
    
    def forward(self, pred, target):
        self.pred = pred
        self.target = target
        self.N = pred.shape[0]
        self.error = pred - target
        abs_error = np.abs(self.error)
        self.is_small = abs_error <= self.delta
        loss = np.where(self.is_small,
                       0.5 * self.error ** 2,
                       self.delta * abs_error - 0.5 * self.delta ** 2)
        return np.mean(loss)
    
    def backward(self):
        grad = np.where(self.is_small,
                       self.error,
                       self.delta * np.sign(self.error))
        return grad / self.N


print("All loss functions defined.")

## 5. Visualize Loss Functions

In [ ]:
# Regression losses: MSE vs Huber
errors = np.linspace(-4, 4, 500).reshape(-1, 1)
target = np.zeros_like(errors)

mse = MSELoss()
huber1 = HuberLoss(delta=1.0)
huber05 = HuberLoss(delta=0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss values
mse_vals = [mse.forward(e.reshape(1,1), np.zeros((1,1))) for e in errors.ravel()]
h1_vals = [huber1.forward(e.reshape(1,1), np.zeros((1,1))) for e in errors.ravel()]
h05_vals = [huber05.forward(e.reshape(1,1), np.zeros((1,1))) for e in errors.ravel()]

ax1.plot(errors.ravel(), mse_vals, label='MSE', linewidth=2)
ax1.plot(errors.ravel(), h1_vals, label='Huber (d=1.0)', linewidth=2, linestyle='--')
ax1.plot(errors.ravel(), h05_vals, label='Huber (d=0.5)', linewidth=2, linestyle=':')
ax1.plot(errors.ravel(), np.abs(errors.ravel()), label='MAE', linewidth=1, linestyle='-', alpha=0.5)
ax1.set_xlabel('Error (pred - target)'); ax1.set_ylabel('Loss')
ax1.set_title('Regression Losses'); ax1.legend(); ax1.grid(True, alpha=0.3)

# Gradients
mse_grads = []; h1_grads = []
for e in errors.ravel():
    mse.forward(e.reshape(1,1), np.zeros((1,1)))
    mse_grads.append(mse.backward().item())
    huber1.forward(e.reshape(1,1), np.zeros((1,1)))
    h1_grads.append(huber1.backward().item())

ax2.plot(errors.ravel(), mse_grads, label='MSE grad', linewidth=2)
ax2.plot(errors.ravel(), h1_grads, label='Huber grad (d=1.0)', linewidth=2, linestyle='--')
ax2.set_xlabel('Error'); ax2.set_ylabel('Gradient')
ax2.set_title('Gradient Comparison (MSE has unbounded grad)'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Huber clips the gradient at delta, making it robust to outliers.")

In [ ]:
# Classification losses: CE vs Focal Loss
p_correct = np.linspace(0.01, 0.99, 200)

ce_loss = -np.log(p_correct)
focal_g2 = -((1 - p_correct) ** 2) * np.log(p_correct)
focal_g5 = -((1 - p_correct) ** 5) * np.log(p_correct)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(p_correct, ce_loss, label='CE (gamma=0)', linewidth=2)
ax1.plot(p_correct, focal_g2, label='Focal (gamma=2)', linewidth=2, linestyle='--')
ax1.plot(p_correct, focal_g5, label='Focal (gamma=5)', linewidth=2, linestyle=':')
ax1.set_xlabel('Predicted probability for correct class')
ax1.set_ylabel('Loss')
ax1.set_title('Cross-Entropy vs Focal Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Show the down-weighting factor
ax2.plot(p_correct, np.ones_like(p_correct), label='CE weight', linewidth=2)
ax2.plot(p_correct, (1 - p_correct) ** 2, label='Focal weight (gamma=2)', linewidth=2, linestyle='--')
ax2.plot(p_correct, (1 - p_correct) ** 5, label='Focal weight (gamma=5)', linewidth=2, linestyle=':')
ax2.set_xlabel('Predicted probability for correct class')
ax2.set_ylabel('Weight')
ax2.set_title('How Focal Loss Down-weights Easy Examples')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Focal loss: easy examples (high p) get near-zero weight => training focuses on hard examples.")

## 6. Numerical Gradient Checking

In [ ]:
def check_activation_grad(act_class, name, x_val=1.5, eps=1e-5, **kwargs):
    act = act_class(**kwargs) if kwargs else act_class()
    
    x = np.array([x_val])
    out = act.forward(x.copy())
    analytical = act.backward(np.ones_like(x))
    
    # Numerical
    act_p = act_class(**kwargs) if kwargs else act_class()
    act_m = act_class(**kwargs) if kwargs else act_class()
    f_plus = act_p.forward(np.array([x_val + eps]))
    f_minus = act_m.forward(np.array([x_val - eps]))
    numerical = (f_plus - f_minus) / (2 * eps)
    
    match = np.allclose(analytical, numerical, atol=1e-4)
    print(f"{name:<15} analytical={analytical[0]:.6f}  numerical={numerical[0]:.6f}  {'OK' if match else 'FAIL'}")

print("Gradient check for activations at x=1.5:")
print("-" * 60)
check_activation_grad(ReLU, 'ReLU')
check_activation_grad(LeakyReLU, 'LeakyReLU', alpha=0.1)
check_activation_grad(ELU, 'ELU')
check_activation_grad(GELU, 'GELU')
check_activation_grad(Swish, 'Swish')
check_activation_grad(Sigmoid, 'Sigmoid')
check_activation_grad(TanhActivation, 'Tanh')

# Also check at negative value
print("\nGradient check for activations at x=-1.0:")
print("-" * 60)
check_activation_grad(ReLU, 'ReLU', x_val=-1.0)
check_activation_grad(LeakyReLU, 'LeakyReLU', x_val=-1.0, alpha=0.1)
check_activation_grad(ELU, 'ELU', x_val=-1.0)
check_activation_grad(GELU, 'GELU', x_val=-1.0)
check_activation_grad(Swish, 'Swish', x_val=-1.0)

## 7. Why GELU/Swish Replaced ReLU in Transformers

Key differences:
- **ReLU** is piecewise linear: exactly 0 for negative inputs. Hard cutoff.
- **GELU/Swish** are smooth, non-monotonic near zero: they allow small negative values through, weighted by magnitude.
- This smoothness helps optimization with Adam (better gradient estimates).
- GELU has a probabilistic interpretation: multiply input by its probability of being positive under a Gaussian.

In [ ]:
x = np.linspace(-3, 3, 500)
relu_out = np.maximum(0, x)
gelu_act = GELU()
gelu_out = gelu_act.forward(x.copy())
swish_act = Swish()
swish_out = swish_act.forward(x.copy())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x, relu_out, label='ReLU', linewidth=2)
ax1.plot(x, gelu_out, label='GELU', linewidth=2, linestyle='--')
ax1.plot(x, swish_out, label='Swish', linewidth=2, linestyle=':')
ax1.axhline(0, color='k', linewidth=0.5)
ax1.set_title('ReLU vs GELU vs Swish')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Zoom in on the interesting region
mask = (x > -2) & (x < 1)
ax2.plot(x[mask], relu_out[mask], label='ReLU', linewidth=2)
ax2.plot(x[mask], gelu_out[mask], label='GELU', linewidth=2, linestyle='--')
ax2.plot(x[mask], swish_out[mask], label='Swish', linewidth=2, linestyle=':')
ax2.axhline(0, color='k', linewidth=0.5)
ax2.set_title('Zoomed: GELU/Swish allow small negative values')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("GELU/Swish: smooth, non-monotonic near 0 => better gradient flow, works well with transformers.")

## 8. Vanishing Gradient Problem — Sigmoid vs Tanh vs ReLU

In [ ]:
# Simulate gradient flow through 20 layers
np.random.seed(42)

def simulate_gradient_flow(activation_fn, n_layers=20, width=64):
    """Track gradient norms through a deep network."""
    # Forward pass
    x = np.random.randn(32, width)
    caches = []
    
    for i in range(n_layers):
        W = np.random.randn(width, width) * np.sqrt(2.0 / width)
        z = x @ W
        act = activation_fn()
        x = act.forward(z)
        caches.append((W, act))
    
    # Backward pass: start with gradient = 1
    grad = np.ones_like(x)
    grad_norms = [np.linalg.norm(grad)]
    
    for W, act in reversed(caches):
        grad = act.backward(grad)
        grad = grad @ W.T
        grad_norms.append(np.linalg.norm(grad))
    
    return grad_norms[::-1]

fig, ax = plt.subplots(figsize=(8, 4))

for act_cls, name, style in [
    (Sigmoid, 'Sigmoid', '-'),
    (TanhActivation, 'Tanh', '--'),
    (ReLU, 'ReLU', '-.'),
    (GELU, 'GELU', ':'),
]:
    norms = simulate_gradient_flow(act_cls)
    ax.plot(norms, label=name, linewidth=2, linestyle=style)

ax.set_yscale('log')
ax.set_xlabel('Layer (from input to output)')
ax.set_ylabel('Gradient norm (log scale)')
ax.set_title('Gradient Flow Through 20 Layers')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Sigmoid: max gradient = 0.25 => multiplied 20 times = 0.25^20 ~ 10^-12 (vanished!)")
print("ReLU: gradient is exactly 1 for positive neurons => no vanishing (but can explode).")

## Interview Quick Reference

**Q: Vanishing gradient with sigmoid/tanh?**  
Sigmoid max gradient is 0.25 (at x=0). Through N layers, gradient shrinks by ~0.25^N. Tanh is better (max grad=1 at x=0) but still saturates. ReLU gradient is exactly 1 for positive inputs — no saturation.

**Q: Why does ReLU work despite non-differentiability at 0?**  
In practice, P(x = exactly 0) = 0. We use a subgradient (usually 0). The discontinuity doesn't cause problems because gradient-based optimization only needs approximately correct gradients.

**Q: Why focal loss for imbalanced data?**  
Standard CE treats all examples equally. In imbalanced settings, the model is "well-calibrated" on majority class (high p) and gets large gradient from minority class misses. Focal loss adds $(1-p)^\gamma$ weight: easy/majority examples with high p get down-weighted, forcing the model to focus on hard/minority examples.

**Q: Softmax + cross-entropy gradient simplifies to (p - y)?**  
Yes. The Jacobian of softmax composed with NLL loss simplifies beautifully: $\frac{\partial L}{\partial z_i} = p_i - \mathbb{1}_{i=y}$. This is numerically stable and elegant — one reason this combination is universal.

**Q: When to use Huber vs MSE?**  
Huber when your data has outliers. MSE has quadratic penalty for large errors, making it sensitive to outliers. Huber transitions to linear (MAE) past delta, bounding the gradient.

---
*Next: 04_backpropagation.ipynb — full manual backprop derivation*